# Figure 1 · Response atlas and representation

**问题：106 株细菌在 13 类神经元上呈现怎样的响应组合？** 本 notebook 从 [00_preparation](00_preparation.ipynb) 生成的动物均值与模板拟合结果出发，逐步说明“曲线 → 时间分箱 → 模板 × 系数 → 跨条件汇总”，并导出主图和方法条。关联的支持图用于理解筛选规则、模板和参数敏感性。

这里读取 `poster/data/prepared/responses/`，本册不重新拟合模板；上游拟合、SNR 和敏感性计算在 00 中展开。主图只展示前 25 s；模型、SNR 和后续相似度使用完整 0–40 s。图中文字为英文，解释为中文。

完成标准：重建值、样本顺序、系数和旧版保存表一致；主图与支持图均有可编辑 SVG、PDF 和 PNG。

In [ ]:
from pathlib import Path
import sys

# Run from the repository root or any directory below it (including poster/notebooks).
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "poster/src/poster_analysis").is_dir() and (p / "reports").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook from this repository or its poster/notebooks directory")
sys.path.insert(0, str(ROOT / "poster/src"))

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from poster_analysis.paths import report_path, prepared_response_path, figure_dir, table_dir
from poster_analysis.style import apply_style, save_figure
from poster_analysis.constants import NEURONS, NEURON_COLORS, SEED

apply_style()
REFERENCE = report_path("responses")
REPORT = prepared_response_path(ROOT)
TABLES = REPORT / "tables"
MAIN, SUPPORT, EXPORT_TABLES = figure_dir("main"), figure_dir("supporting"), table_dir()
np.random.seed(SEED)

from poster_analysis.atlas import (
    load_atlas_data, aggregate_conditions, plot_response_atlas,
    plot_method_strip, plot_template_atlas,
)

# Display parameters only; changing these does not refit the saved analysis.
DISPLAY_BINS = 5
PROFILE_LIMITS = (-0.5, 1.0)
AMPLITUDE_LIMITS = (-0.6, 0.6)
EXAMPLE_SAMPLE, EXAMPLE_NEURON, EXAMPLE_BLOCK = "A300", "AWCON", "20260520"

## 1. 来源、维度与覆盖

主要输入由 00 从已审核个体曲线计算并写入准备目录；历史显示表仅用于独立核对。未准备或文件校验失败时会明确要求先运行 00。`representation_arrays.npz` 按 **condition × neuron × time** 存储；condition 是一次样本与记录批次的组合。一个样本有多个条件时，按照既有规则对可用条件等权平均。它是汇总规则，不是本图要解释的生物变量。

In [ ]:
sources = pd.DataFrame([
    ("Saved means, SNR, templates and reconstructions", "tables/representation_arrays.npz"),
    ("Strain coefficients", "tables/strain_coefficients.csv"),
    ("Fixed row order from original raw-profile hierarchy", "tables/figure_row_order.csv"),
    ("Prepared five-bin display", "tables/display_profile_5bin.csv"),
    ("Condition coverage, gate and coefficients", "tables/condition_metrics.csv"),
    ("Strain-neuron gate states", "tables/filter_state_by_strain_cell.csv"),
    ("Gate and aggregation definitions", "representation_parameters.json"),
    ("Saved threshold sensitivity", "tables/sensitivity_metrics.csv"),
], columns=["Role", "File relative to prepared inputs"])
import hashlib
sources['Source relative to repository'] = [str((REPORT / name).relative_to(ROOT)) for name in sources['File relative to prepared inputs']]
sources['SHA256'] = [hashlib.sha256((ROOT / name).read_bytes()).hexdigest() for name in sources['Source relative to repository']]
reference_path = REFERENCE / 'tables/display_profile_5bin.csv'
sources.loc[len(sources)] = ['Historical display for independent verification', '', str(reference_path.relative_to(ROOT)), hashlib.sha256(reference_path.read_bytes()).hexdigest()]
display(sources[['Role', 'Source relative to repository']])
print('Prepared input directory:', REPORT.relative_to(ROOT))

data = load_atlas_data(REPORT)
a = data["arrays"]
neurons, order = data["neurons"], data["sample_order"]
assert neurons == NEURONS
shape_summary = pd.DataFrame([
    (name, str(a[name].shape), str(a[name].dtype))
    for name in ["means", "mean_bins", "templates", "coefficients", "reconstruction"]
], columns=["Array", "Shape", "dtype"])
display(shape_summary)

In [ ]:
conditions = data["conditions"]
support = conditions.groupby("cell").agg(
    n_conditions=("strain", "size"), n_strains=("strain", "nunique"),
    min_animals=("n_animals", "min"), median_animals=("n_animals", "median"),
    max_animals=("n_animals", "max"),
).reindex(neurons)
status_counts = pd.crosstab(conditions.cell, conditions.status).reindex(neurons).fillna(0).astype(int)
support = support.join(status_counts)
display(support)
support.to_csv(EXPORT_TABLES / "fig01_sample_support.csv")

fig, ax = plt.subplots(figsize=(10, 3.4), layout="constrained")
status_counts.plot.bar(stacked=True, ax=ax, color=["#D4DCE5", "#467BA5"], width=.8)
ax.set(xlabel="Neuron class", ylabel="Condition × neuron entries")
ax.tick_params(axis="x", rotation=0)
ax.legend(title="Saved gate status", frameon=False, ncols=2)
save_figure(fig, SUPPORT / "fig01_condition_support")
plt.show()

## 2. 最小示例：40 个一秒均值如何变成八个时间箱

取此前 poster 已使用的 **A300–AWCON** 示例，仅用于展示计算过程。该示例按此前的强响应示例选择，并非随机、典型或独立验证样本。先对每连续 5 个一秒均值取平均；保存的模板 $h_c$ 在每类神经元内共享，满足 RMS = 1。系数 $a_{kc}$ 乘模板给出近似重建 $\hat y_{kct}=a_{kc}h_{ct}$。

模板与系数不是从这条示例独立估计出来的；图中同时画出原始均值、分箱均值和模型重建，以保留近似误差。

In [ ]:
k = np.flatnonzero((a["conditions"][:, 0] == EXAMPLE_SAMPLE) &
                   (a["conditions"][:, 1] == EXAMPLE_BLOCK)).item()
c = neurons.index(EXAMPLE_NEURON)
mean_curve = a["means"][k, c]
bin_means = mean_curve.reshape(8, 5).mean(axis=1)
template = a["templates"][c]
coefficient = a["coefficients"][k, c]
reconstructed_example = coefficient * template
np.testing.assert_allclose(bin_means, a["mean_bins"][k, c], rtol=1e-12, atol=1e-14)
np.testing.assert_allclose(reconstructed_example, a["reconstruction"][k, c], rtol=1e-12, atol=1e-14)
np.testing.assert_allclose(np.sqrt((a["templates"] ** 2).mean(axis=1)), 1.0, atol=1e-12)
display(pd.DataFrame({"bin_start_s": np.arange(0, 40, 5), "observed_mean": bin_means,
                      "template": template, "coefficient_times_template": reconstructed_example}))
print(f"Example: {EXAMPLE_SAMPLE} / {EXAMPLE_NEURON}; animals = {a['counts'][k,c]}; coefficient = {coefficient:.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(7.4, 3.3), layout="constrained")
ax.plot(np.arange(40), mean_curve, color="#333333", lw=1.5, label="Saved animal-mean response")
ax.plot(np.arange(2.5, 40, 5), bin_means, "o-", ms=4, label="Eight observed bin means")
ax.plot(np.arange(2.5, 40, 5), reconstructed_example, "s--", ms=4,
        color="#B2182B", label="Amplitude × shared template")
ax.axvspan(0, 10, color="#D8DEE5", alpha=.4)
ax.axhline(0, color="gray", lw=.6)
ax.set(xlabel="Time (s)", ylabel=r"$\Delta F/F_0$", title=f"{EXAMPLE_SAMPLE} · {EXAMPLE_NEURON}")
ax.legend(frameon=False, fontsize=9)
save_figure(fig, SUPPORT / "fig01_observed_vs_reconstructed_example")
plt.show()

## 3. 在 notebook 中重建完整响应图谱

先在条件层面相乘，再在每株细菌内对有限的可用条件等权平均。筛选后的零保留为零；缺失保留为 `NaN`。以下代码显式展示这一步，并与提取出的通用聚合函数、保存的系数和五箱显示表交叉核对。

样本排序沿用已保存的 **未筛选完整 0–40 s 原始曲线距离的 average-linkage 层次**。此处没有按照 poster 图重新挑选或排序样本。

In [ ]:
condition_reconstruction = a["coefficients"][:, :, None] * a["templates"][None, :, :]
np.testing.assert_allclose(condition_reconstruction, a["reconstruction"], equal_nan=True, atol=1e-14)

# Explicit finite-value, equal-condition aggregation, with no imputation.
strain_reconstruction = []
for sample in order:
    selected = condition_reconstruction[a["conditions"][:, 0] == sample]
    finite = np.isfinite(selected)
    total = np.where(finite, selected, 0.0).sum(axis=0)
    count = finite.sum(axis=0)
    strain_reconstruction.append(np.divide(total, count, out=np.full(total.shape, np.nan), where=count > 0))
strain_reconstruction = np.asarray(strain_reconstruction)
np.testing.assert_allclose(strain_reconstruction,
    aggregate_conditions(a["reconstruction"], a["conditions"], order), equal_nan=True, atol=1e-14)

amplitudes = aggregate_conditions(a["coefficients"], a["conditions"], order)
np.testing.assert_allclose(amplitudes, data["coefficients"].loc[order], equal_nan=True, atol=1e-14)
profile = strain_reconstruction[:, :, :DISPLAY_BINS]
previous = pd.read_csv(REFERENCE / "tables/display_profile_5bin.csv", index_col=0).loc[order].to_numpy()
np.testing.assert_allclose(profile.reshape(len(order), -1), previous, equal_nan=True, atol=1e-14)
print("Verified exact saved representation:", profile.shape, "(sample × neuron × displayed bin)")

## 4. Poster 主图与方法条

主图覆盖全部 106 株、13 类神经元，展示前五个 5-s 箱。虚线标记刺激在 10 s 结束。颜色是模型重建的 $\Delta F/F_0$，并非神经元分别 z-score 后的值。白色接近零，但筛选为零仅是分析规则，不代表生理无响应。灰色代表缺失；色条范围外的值只在显示中饱和。

方法条适合放在主图附近，其数值来自上面的可核对示例。完整模板及幅度矩阵放到支持图，避免主图重复。

In [ ]:
fig = plot_response_atlas(profile, neurons, order, limits=PROFILE_LIMITS)
save_figure(fig, MAIN / "fig01_response_atlas")
plt.show()

fig = plot_method_strip(mean_curve, bin_means, template, coefficient,
                        label=f"{EXAMPLE_NEURON} · {EXAMPLE_SAMPLE}")
save_figure(fig, MAIN / "fig01_method_strip")
plt.show()

In [ ]:
fig = plot_template_atlas(a["templates"], amplitudes, neurons, order, limits=AMPLITUDE_LIMITS)
save_figure(fig, SUPPORT / "fig01_templates_and_amplitudes")
plt.show()

## 5. 支持探索：SNR 的含义与阈值敏感性

既有筛选使用单只动物先做 trial 平均后的曲线：$P=\operatorname{mean}_t(\mu_t^2)$，$V$ 是跨动物样本方差再对时间求均值；$C=P-V/n$，$\mathrm{SNR}=\sqrt{\max(C,0)/V}$。这里从保存的功率量复核数值，不重新估计原始动物方差。

主阈值 0.5 是已有探索设置；下面 0.25、0.5、0.75、1.0 的曲线只说明保留数量如何变化，没有根据神经—化学相关优化阈值。缺失或不足两只动物不应当视为筛选后的零。

In [ ]:
P, V, n = a["signal_power"], a["scatter_power"], a["counts"]
C = P - np.divide(V, n, out=np.full_like(V, np.nan), where=n > 0)
ratio = np.divide(np.maximum(C, 0), V, out=np.zeros_like(V), where=V > 0)
snr_recomputed = np.sqrt(ratio)
snr_recomputed[(V == 0) & (C > 0)] = np.inf
adequate = (n >= 2) & np.isfinite(P) & np.isfinite(V)
np.testing.assert_allclose(snr_recomputed[adequate], a["snr"][adequate], rtol=1e-12, atol=1e-14)

sensitivity = data["sensitivity"].copy()
summary = sensitivity.groupby("threshold")[["eligible", "retained", "suppressed", "missing", "limited_n"]].sum()
display(summary)
summary.to_csv(EXPORT_TABLES / "fig01_threshold_sensitivity.csv")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5), layout="constrained")
finite_snr = a["snr"][np.isfinite(a["snr"])]
axes[0].hist(np.log10(1 + finite_snr), bins=30, color="#467BA5", alpha=.75)
axes[0].axvline(np.log10(1.5), color="#B2182B", ls="--", label="SNR = 0.5")
axes[0].set(xlabel="log10(1 + individual SNR)", ylabel="Condition × neuron entries")
axes[0].legend(frameon=False)
for cell in neurons:
    rows = sensitivity.loc[sensitivity.cell.eq(cell) & sensitivity.threshold.ne("unfiltered")].copy()
    rows["cutoff"] = pd.to_numeric(rows.threshold)
    rows = rows.sort_values("cutoff")
    axes[1].plot(rows.cutoff, rows.retained / rows.eligible, "o-", ms=3, lw=1, color=NEURON_COLORS[cell], label=cell)
axes[1].set(xlabel="Individual-SNR cutoff", ylabel="Retained / eligible", ylim=(0, 1.05))
axes[1].legend(frameon=False, fontsize=7, ncols=4, loc="upper center", bbox_to_anchor=(.5, -.24))
save_figure(fig, SUPPORT / "fig01_snr_sensitivity")
plt.show()

## 6. 读图边界与输出

这张图提供神经响应组合的描述，并未证明这些模板具有因果机制或分类能力。模板 RMS = 1，最大绝对值箱固定为正，因此系数正负依赖模板方向；不能统一解释为兴奋或抑制。全数据图谱中的模板是描述性拟合，跨动物验证见 `02_repeatability.ipynb`。

下面记录显示饱和数量、样本量与核对结果。所有新输出写入 `poster/figures` 和 `poster/tables`，历史报告中的数据与图不被覆盖。

In [ ]:
checks = {
    "n_strains": len(order), "n_neurons": len(neurons), "n_conditions": len(a["conditions"]),
    "display_bins": DISPLAY_BINS, "fit_bins": a["templates"].shape[1],
    "profile_saturated_bins": int(((profile < PROFILE_LIMITS[0]) | (profile > PROFILE_LIMITS[1])).sum()),
    "amplitude_saturated_entries": int(((amplitudes < AMPLITUDE_LIMITS[0]) | (amplitudes > AMPLITUDE_LIMITS[1])).sum()),
    "profile_max_abs_difference_from_saved": float(np.nanmax(np.abs(profile.reshape(len(order), -1) - previous))),
    "coefficient_and_template_reconstruction_verified": True,
    "prepared_inputs": str(REPORT.relative_to(ROOT)),
    "reference_report": str(REFERENCE.relative_to(ROOT)),
    "prepared_by": "poster/notebooks/00_preparation.ipynb", "refit_here": False,
}
(EXPORT_TABLES / "fig01_checks.json").write_text(json.dumps(checks, indent=2) + "\n")
sources.to_csv(EXPORT_TABLES / "fig01_sources.csv", index=False)
display(pd.Series(checks, name="Value").to_frame())